# Tabelas Delta: Criação, Constraints e Clonagem

Este documento resume os principais conceitos para configuração de tabelas Delta Lake no Azure Databricks, abordando CTAS statements, Table Constraints e Cloning de tabelas.

---

## 1. CTAS (CREATE TABLE AS SELECT)

O comando **CTAS** permite criar uma nova tabela a partir de uma consulta SELECT em uma tabela existente. É uma forma eficiente de criar subconjuntos ou transformações de dados.

### Sintaxe Básica

```sql
CREATE TABLE table_1
AS SELECT * FROM table_2
```

### Características Principais

- **Inferência automática de schema**: O schema é derivado automaticamente dos resultados da query
- **Não suporta declaração manual de schema**: Diferente do `CREATE TABLE` tradicional
- **Tabela criada com dados**: A tabela já nasce populada com os dados da consulta

### Filtrando e Renomeando Colunas

```sql
CREATE TABLE table_1
AS SELECT col_1, col_3 AS new_col_3 FROM table_2
```

### Opções Adicionais

O CTAS suporta diversas opções como comentários, particionamento e localização customizada:

```sql
CREATE TABLE new_table
COMMENT "Contains PII"
PARTITIONED BY (city, birth_date)
LOCATION '/some/path'
AS SELECT id, name, email, birth_date, city FROM users
```

### CREATE TABLE vs CTAS - Comparativo

| Aspecto | CREATE TABLE | CTAS |
|---------|--------------|------|
| Schema | Declaração manual | Inferência automática |
| Dados | Tabela vazia | Tabela com dados |
| Inserção | Requer `INSERT INTO` | Dados inseridos na criação |

> 📘 **Da documentação Azure**: O CTAS é particularmente útil para converter arquivos raw (CSV, JSON, Parquet) em tabelas Delta, aproveitando a inferência automática de schema e tipos de dados. No entanto, não é possível definir identity columns ou outras especificações de tabela usando CTAS - para esses casos, crie a tabela com schema completo e use INSERT.

---

## 2. Table Constraints

As constraints garantem integridade e qualidade dos dados nas tabelas Delta. Quando uma constraint é violada, a transação falha com um erro `InvariantViolationException`.

### Tipos de Constraints Suportadas

#### NOT NULL Constraint

Indica que valores em colunas específicas não podem ser nulos. É definida no schema durante a criação da tabela:

```sql
CREATE TABLE people10m (
  id INT NOT NULL,
  firstName STRING,
  middleName STRING NOT NULL,
  lastName STRING,
  gender STRING,
  birthDate TIMESTAMP,
  ssn STRING,
  salary INT
);

-- Remover constraint NOT NULL
ALTER TABLE people10m ALTER COLUMN middleName DROP NOT NULL;

-- Adicionar constraint NOT NULL
ALTER TABLE people10m ALTER COLUMN ssn SET NOT NULL;
```

#### CHECK Constraint

Avalia uma expressão booleana para cada linha de entrada. Os dados só são aceitos se a expressão retornar `TRUE`:

```sql
-- Sintaxe geral
ALTER TABLE table_name ADD CONSTRAINT constraint_name constraint_details

-- Exemplo prático
ALTER TABLE orders ADD CONSTRAINT valid_date CHECK (date > '2020-01-01');

-- Outro exemplo
ALTER TABLE people10m ADD CONSTRAINT validIds CHECK (id > 1 AND id < 99999999);

-- Remover constraint
ALTER TABLE people10m DROP CONSTRAINT validIds;
```

> ⚠️ **Importante**: Ao adicionar uma constraint, o Azure Databricks verifica automaticamente se todos os registros existentes satisfazem a condição. Se houver violação, um erro será levantado.

### Visualizando Constraints

```sql
-- Ver detalhes da tabela incluindo constraints
DESCRIBE DETAIL people10m;

-- Ver propriedades da tabela
SHOW TBLPROPERTIES people10m;
```

### Primary Key e Foreign Key (Informational)

A partir do Databricks Runtime 11.3 LTS, é possível definir chaves primárias e estrangeiras. Porém, são **informacionais** e não são enforced automaticamente:

```sql
CREATE TABLE T(
  pk1 INTEGER NOT NULL, 
  pk2 INTEGER NOT NULL, 
  CONSTRAINT t_pk PRIMARY KEY(pk1, pk2)
);

CREATE TABLE S(
  pk INTEGER NOT NULL PRIMARY KEY, 
  fk1 INTEGER, 
  fk2 INTEGER, 
  CONSTRAINT s_t_fk FOREIGN KEY(fk1, fk2) REFERENCES T
);
```

---

## 3. Cloning Delta Lake Tables

O Delta Lake oferece dois tipos de clonagem para criar cópias de tabelas, cada um com características específicas.

### Deep Clone

Cria uma **cópia completa e independente** dos dados e metadados da tabela de origem.

```sql
CREATE TABLE table_clone
DEEP CLONE source_table
```

**Características:**
- Copia dados + metadados completamente
- Pode sincronizar mudanças incrementalmente
- Pode demorar bastante para datasets grandes
- Clone totalmente independente da tabela fonte
- Ideal para: backup, disaster recovery, arquivamento de dados

**Sintaxe com opções:**

```sql
-- Clone básico (DEEP é o padrão)
CREATE TABLE target_table CLONE source_table;

-- Substituir tabela existente
CREATE OR REPLACE TABLE target_table CLONE source_table;

-- Criar apenas se não existir
CREATE TABLE IF NOT EXISTS target_table CLONE source_table;
```

### Shallow Clone

Cria uma cópia **rápida** da tabela, copiando apenas os logs de transação Delta (metadados).

```sql
CREATE TABLE table_clone
SHALLOW CLONE source_table
```

**Características:**
- Copia apenas metadados (Delta transaction logs)
- Muito rápido para criar
- Custo mínimo de armazenamento
- **Depende dos arquivos de dados da tabela fonte**
- Se executar VACUUM na fonte, o clone pode ficar inutilizável
- Ideal para: testes, experimentação, ambientes de desenvolvimento

**Sintaxe com Time Travel:**

```sql
-- Clone de versão específica
CREATE TABLE target_table SHALLOW CLONE source_table VERSION AS OF 5;

-- Clone de timestamp específico
CREATE TABLE target_table SHALLOW CLONE source_table 
TIMESTAMP AS OF '2024-01-01';

-- Timestamp relativo
CREATE TABLE target_table SHALLOW CLONE source_table 
TIMESTAMP AS OF date_sub(current_date(), 1);
```

### Comparativo Deep Clone vs Shallow Clone

| Aspecto | Deep Clone | Shallow Clone |
|---------|------------|---------------|
| Dados copiados | Sim | Não |
| Metadados copiados | Sim | Sim |
| Velocidade | Lento (proporcional ao tamanho) | Rápido |
| Independência | Total | Depende da fonte |
| Custo de storage | Alto (duplica dados) | Mínimo |
| Caso de uso | Produção, DR, arquivo | Testes, dev |

### Metadados Clonados

Os seguintes metadados são copiados em ambos os tipos:
- Schema
- Informações de particionamento
- Invariants
- Nullability

**Apenas em Deep Clone:**
- Metadados de streaming
- Metadados de COPY INTO

**Não são clonados:**
- Descrição da tabela
- Commit metadata definido pelo usuário

> 💡 **Casos de Uso**: Clone é extremamente útil para configurar tabelas de teste em ambiente de desenvolvimento. Em ambos os casos, **modificações no clone não afetam a tabela de origem**.

---

## Links para Consulta

### Documentação Oficial Microsoft/Azure Databricks

- [CREATE TABLE [USING]](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-create-table-using) - Sintaxe completa do CREATE TABLE
- [Clone a table on Azure Databricks](https://learn.microsoft.com/en-us/azure/databricks/delta/clone) - Guia completo sobre clonagem
- [CREATE TABLE CLONE](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-clone) - Referência SQL para CLONE
- [Constraints on Azure Databricks](https://learn.microsoft.com/en-us/azure/databricks/tables/constraints) - Guia de constraints
- [ADD CONSTRAINT clause](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-alter-table-add-constraint) - Referência SQL para constraints
- [Delta Lake generated columns](https://learn.microsoft.com/en-us/azure/databricks/delta/generated-columns) - Colunas geradas automaticamente
- [Convert to Delta Lake](https://learn.microsoft.com/en-us/azure/databricks/ingestion/data-migration/convert-to-delta) - Conversão de Parquet/Iceberg para Delta
- [What is Delta Lake?](https://learn.microsoft.com/en-us/azure/databricks/delta/) - Visão geral do Delta Lake
- [Best practices: Delta Lake](https://learn.microsoft.com/en-us/azure/databricks/delta/best-practices) - Melhores práticas

### Databricks Blog

- [Easily Clone your Delta Lake for Testing, Sharing, and ML Reproducibility](https://www.databricks.com/blog/2020/09/15/easily-clone-your-delta-lake-for-testing-sharing-and-ml-reproducibility.html)
- [Delta Clones Simplify Disaster Recovery](https://www.databricks.com/blog/2021/04/20/attack-of-the-delta-clones-against-disaster-recovery-availability-complexity.html)

---

## Diagrama: Fluxo de Criação de Tabelas

```
┌─────────────────────────────────────────────────────────────────┐
│                    Criação de Tabelas Delta                     │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐      │
│  │ CREATE TABLE │    │    CTAS      │    │    CLONE     │      │
│  │  (Manual)    │    │ (Auto-infer) │    │  (Cópia)     │      │
│  └──────┬───────┘    └──────┬───────┘    └──────┬───────┘      │
│         │                   │                   │               │
│         ▼                   ▼                   ▼               │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐      │
│  │ Schema       │    │ Schema       │    │ Deep Clone   │      │
│  │ declarado    │    │ inferido     │    │ (completo)   │      │
│  │ manualmente  │    │ da query     │    │      ou      │      │
│  │              │    │              │    │ Shallow Clone│      │
│  │ Tabela vazia │    │ Com dados    │    │ (metadados)  │      │
│  └──────┬───────┘    └─────────────-┘    └──────────────┘      │
│         │                                                       │
│         ▼                                                       │
│  ┌──────────────┐                                              │
│  │ INSERT INTO  │                                              │
│  │ para popular │                                              │
│  └──────────────┘                                              │
│                                                                 │
├─────────────────────────────────────────────────────────────────┤
│                      Constraints                                │
│  ┌──────────────┐    ┌──────────────┐    ┌──────────────┐      │
│  │  NOT NULL    │    │    CHECK     │    │  PK / FK     │      │
│  │  (enforced)  │    │  (enforced)  │    │(informational│      │
│  └──────────────┘    └──────────────┘    └──────────────┘      │
└─────────────────────────────────────────────────────────────────┘
```

---

## Resumo Rápido

| Conceito | Comando Principal | Uso |
|----------|-------------------|-----|
| CTAS | `CREATE TABLE ... AS SELECT` | Criar tabela com dados de query |
| NOT NULL | `ALTER TABLE ... ALTER COLUMN ... SET NOT NULL` | Garantir valores não nulos |
| CHECK | `ALTER TABLE ... ADD CONSTRAINT ... CHECK (...)` | Validar expressões booleanas |
| Deep Clone | `CREATE TABLE ... DEEP CLONE ...` | Cópia completa independente |
| Shallow Clone | `CREATE TABLE ... SHALLOW CLONE ...` | Cópia rápida de metadados |

---